In [16]:
import os
base = '/content/bcn20000/bcn_20k_train'

# List top-level contents
print("Top-level contents:")
for item in os.listdir(base):
    print("  ", item)

# Look for CSV
csvs = [f for f in os.listdir(base) if f.endswith('.csv')]
if csvs:
    print(f"\n✅ CSV found: {csvs[0]}")
else:
    # Check subfolders
    for root, dirs, files in os.walk(base):
        for f in files:
            if f.endswith('.csv'):
                print(f"\n✅ CSV found in subfolder: {os.path.join(root, f)}")
                break
        else:
            continue
        break

# Look for images
exts = ('.jpg', '.jpeg', '.png')
img_count = 0
for root, dirs, files in os.walk(base):
    for f in files:
        if f.lower().endswith(exts):
            img_count += 1
            if img_count == 1:
                print(f"\n✅ Image folder: {root}")
    if img_count > 0:
        break
print(f"   (found {img_count} images in first directory)")

Streaming output truncated to the last 5000 lines.
   BCN_0000018120.jpg
   BCN_0000002911.jpg
   BCN_0000009672.jpg
   BCN_0000000784.jpg
   BCN_0000010111.jpg
   BCN_0000011001.jpg
   BCN_0000017708.jpg
   BCN_0000003392.jpg
   BCN_0000008636.jpg
   BCN_0000010918.jpg
   BCN_0000005265.jpg
   BCN_0000003167.jpg
   BCN_0000017573.jpg
   BCN_0000017129.jpg
   BCN_0000011618.jpg
   BCN_0000018111.jpg
   BCN_0000010773.jpg
   BCN_0000011694.jpg
   BCN_0000018047.jpg
   BCN_0000001491.jpg
   BCN_0000004817.jpg
   BCN_0000015089.jpg
   BCN_0000006224.jpg
   BCN_0000003867.jpg
   BCN_0000000532.jpg
   BCN_0000008658.jpg
   BCN_0000016343.jpg
   BCN_0000014403.jpg
   BCN_0000017506.jpg
   BCN_0000013706.jpg
   BCN_0000019118.jpg
   BCN_0000008284.jpg
   BCN_0000007650.jpg
   BCN_0000002523.jpg
   BCN_0000017562.jpg
   BCN_0000014379.jpg
   BCN_0000006771.jpg
   BCN_0000003052.jpg
   BCN_0000005699.jpg
   BCN_0000013220.jpg
   BCN_0000016815.jpg
   BCN_0000002059.jpg
   BCN_0000014681.jpg
   

In [21]:
# ============================================================
# ViT-B/16 Baseline Training for BCN20000 (Corrected)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.preprocessing import label_binarize
import timm
from tqdm.notebook import tqdm
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. Paths
# ============================================================
CSV_PATH = '/content/drive/My Drive/SKIN paper/bcn_20k_train.csv'
IMAGE_DIR = '/content/bcn20000/bcn_20k_train'

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
if not os.path.exists(IMAGE_DIR):
    raise FileNotFoundError(f"Image dir not found: {IMAGE_DIR}")

print(f"✅ Metadata CSV: {CSV_PATH}")
print(f"✅ Image directory: {IMAGE_DIR}")

# ============================================================
# 2. Load metadata
# ============================================================
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from CSV")
print("Columns:", df.columns.tolist())

# Explicitly set column names (known from your CSV)
diag_col = 'diagnosis'
img_col = 'bcn_filename'
lesion_col = 'lesion_id'

print(f"Diagnosis column: {diag_col}")
print(f"Image column: {img_col}")
print(f"Lesion column: {lesion_col}")

# ============================================================
# 3. Map diagnosis names to integer labels (0..5)
# ============================================================
label_map = {
    'ak': 0, 'actinic keratosis': 0,
    'bcc': 1, 'basal cell carcinoma': 1,
    'mel': 2, 'melanoma': 2,
    'met': 3, 'melanoma metastasis': 3,
    'nv': 4, 'nevus': 4,
    'sk': 5, 'seborrheic keratosis': 5,
}

df['label'] = df[diag_col].astype(str).str.lower().str.strip().map(label_map)

# Drop rows with unknown labels (SCC, BKL, DF, VASC, etc.)
unknown = df[df['label'].isna()]
if len(unknown) > 0:
    print(f"⚠️ Dropping {len(unknown)} rows with unknown labels: {unknown[diag_col].unique()}")
    df = df.dropna(subset=['label']).reset_index(drop=True)

df['label'] = df['label'].astype(int)
print("\nClass distribution (after dropping unknowns):")
print(df['label'].value_counts().sort_index())

# ============================================================
# 4. Build image paths
# ============================================================
df['image_path'] = df[img_col].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Fix missing extensions (some images may have .jpg, .jpeg, .png)
def fix_path(p):
    if os.path.exists(p):
        return p
    base, ext = os.path.splitext(p)
    for e in ['.jpg', '.jpeg', '.png']:
        cand = base + e
        if os.path.exists(cand):
            return cand
    return p

df['image_path'] = df['image_path'].apply(fix_path)

# Drop rows where image file doesn't exist
missing = df[~df['image_path'].apply(os.path.exists)]
if len(missing) > 0:
    print(f"⚠️ {len(missing)} images missing. Dropping.")
    df = df[df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f"Final dataset size: {len(df)} samples")

# ============================================================
# 5. Split data (lesion-level)
# ============================================================
def split_data(df, lesion_col, test_size=0.1, val_size=0.1, random_state=42):
    lesions = df.groupby(lesion_col).agg({'label': 'first'}).reset_index()
    train_val_lesions, test_lesions = train_test_split(
        lesions, test_size=test_size, stratify=lesions['label'],
        random_state=random_state
    )
    val_ratio = val_size / (1 - test_size)
    train_lesions, val_lesions = train_test_split(
        train_val_lesions, test_size=val_ratio,
        stratify=train_val_lesions['label'],
        random_state=random_state
    )
    train_df = df[df[lesion_col].isin(train_lesions[lesion_col])].reset_index(drop=True)
    val_df = df[df[lesion_col].isin(val_lesions[lesion_col])].reset_index(drop=True)
    test_df = df[df[lesion_col].isin(test_lesions[lesion_col])].reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = split_data(df, lesion_col)
print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Train class distribution:")
print(train_df['label'].value_counts().sort_index())

# ============================================================
# 6. Dataset class
# ============================================================
class BCN20000Dataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.df.iloc[idx]['label']

# ============================================================
# 7. Transforms
# ============================================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ============================================================
# 8. DataLoaders
# ============================================================
BATCH_SIZE = 32   # reduce to 16 if OOM

train_loader = DataLoader(BCN20000Dataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(BCN20000Dataset(val_df, val_transform),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(BCN20000Dataset(test_df, val_transform),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready (batch size {BATCH_SIZE})")

# ============================================================
# 9. Model, Optimizer, Loss
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=6)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-6)

print(f"✅ Model: ViT-B/16 with {sum(p.numel() for p in model.parameters()):,} params")

# ============================================================
# 10. Training Loop
# ============================================================
NUM_EPOCHS = 30
best_val_loss = float('inf')
best_state = None
patience = 5
wait = 0

print("\n" + "="*60)
print("Starting Training")
print("="*60)

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'Loss': f'{loss.item():.4f}'})

    train_acc = correct / total
    train_loss = train_loss / total

    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    val_mcc = matthews_corrcoef(all_labels, all_preds)

    print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Loss={train_loss:.4f} | Val Acc={val_acc:.4f}, Loss={val_loss:.4f}, F1={val_f1:.4f}, MCC={val_mcc:.4f}")

    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict().copy()
        wait = 0
        print("  ✅ Best model updated")
    else:
        wait += 1
        if wait >= patience:
            print(f"  ⏹️ Early stopping after {epoch+1} epochs")
            break

if best_state:
    model.load_state_dict(best_state)

print("\n✅ Training completed.")

# ============================================================
# 11. Test Evaluation
# ============================================================
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1_macro = f1_score(all_labels, all_preds, average='macro')
mcc = matthews_corrcoef(all_labels, all_preds)
all_labels_onehot = label_binarize(all_labels, classes=list(range(6)))
roc = roc_auc_score(all_labels_onehot, np.array(all_probs), average='macro', multi_class='ovr')
f1_per_class = f1_score(all_labels, all_preds, average=None)

print("\n" + "="*60)
print("TEST RESULTS")
print("="*60)
print(f"Accuracy  : {acc:.4f}")
print(f"Macro-F1  : {f1_macro:.4f}")
print(f"MCC       : {mcc:.4f}")
print(f"ROC-AUC   : {roc:.4f}")
print(f"Per-class F1: {f1_per_class}")
print("="*60)

# ============================================================
# 12. Save Results
# ============================================================
results_df = pd.DataFrame({
    'model': ['ViT-B/16'],
    'accuracy': [acc],
    'macro_f1': [f1_macro],
    'mcc': [mcc],
    'roc_auc': [roc]
})
results_df.to_csv('/content/vit_b16_results.csv', index=False)
torch.save(best_state, '/content/vit_b16_best_model.pth')

try:
    !cp /content/vit_b16_results.csv "/content/drive/My Drive/SKIN paper/"
    !cp /content/vit_b16_best_model.pth "/content/drive/My Drive/SKIN paper/"
    print("\n✅ Results saved to: /content/drive/My Drive/SKIN paper/")
except:
    print("\n✅ Results saved locally at /content/")

print("\n📊 vit_b16_results.csv")
print("🧠 vit_b16_best_model.pth")
print("\n🎉 ViT-B/16 baseline training complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Metadata CSV: /content/drive/My Drive/SKIN paper/bcn_20k_train.csv
✅ Image directory: /content/bcn20000/bcn_20k_train
Loaded 12413 rows from CSV
Columns: ['bcn_filename', 'age_approx', 'anatom_site_general', 'diagnosis', 'lesion_id', 'capture_date', 'sex', 'split']
Diagnosis column: diagnosis
Image column: bcn_filename
Lesion column: lesion_id
⚠️ Dropping 1804 rows with unknown labels: ['SCC' 'BKL' 'DF' 'VASC']

Class distribution (after dropping unknowns):
label
0     737
1    2809
2    2857
4    4206
Name: count, dtype: int64
Final dataset size: 10609 samples

Train: 8484, Val: 1085, Test: 1040
Train class distribution:
label
0     585
1    2231
2    2319
4    3349
Name: count, dtype: int64
✅ DataLoaders ready (batch size 32)
Using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

✅ Model: ViT-B/16 with 85,803,270 params

Starting Training


Epoch 1/30:   0%|          | 0/266 [00:00<?, ?it/s]

...

Epoch 12: Train Acc=0.7135, Loss=0.6996 | Val Acc=0.6793, Loss=0.8084, F1=0.5952, MCC=0.5413


Epoch 13/30:   0%|          | 0/266 [00:00<?, ?it/s]

Epoch 13: Train Acc=0.7212, Loss=0.6811 | Val Acc=0.6783, Loss=0.8910, F1=0.5913, MCC=0.5598
  ⏹️ Early stopping after 13 epochs

✅ Training completed.


Testing:   0%|          | 0/33 [00:00<?, ?it/s]


TEST RESULTS
Accuracy  : 0.6462
Macro-F1  : 0.5221
MCC       : 0.4857
ROC-AUC   : nan
Per-class F1: [0.16666667 0.68956407 0.50126582 0.73076923]

✅ Results saved to: /content/drive/My Drive/SKIN paper/

📊 vit_b16_results.csv
🧠 vit_b16_best_model.pth

🎉 ViT-B/16 baseline training complete!
